In [ ]:
import datania
import pandas as pd
import os

# Create a directory for our "large" dataset
os.makedirs('data/agricultural_survey', exist_ok=True)

# Generate 5 years of data, saved as separate Parquet files
for year in range(2021, 2026):
    csv_file = datania.generate_agricultural_survey(n_farms=10000, year=year, seed=year)
    df = pd.read_csv(csv_file)
    df.to_parquet(f'data/agricultural_survey/daps_{year}.parquet', index=False)
    print(f"Created DAPS {year}: {len(df)} records")

In [ ]:
import dask.dataframe as dd
import pandas as pd
import os
import time

# First, run the setup code above to create the data files

# Task 1: Load all files with Dask
print("=== LOADING WITH DASK ===")

# YOUR CODE HERE: Use glob pattern to load all parquet files
# Hint: dd.read_parquet('data/agricultural_survey/*.parquet')
ddf = None  # Replace this

print(f"Number of partitions: {ddf.npartitions}")
print(f"Columns: {ddf.columns.tolist()}")

# This should show a Dask DataFrame, not the actual data
print(f"\nType: {type(ddf)}")
print("Notice: Dask hasn't loaded any data yet!")


# Task 2: Compute summary statistics
print("\n=== MEAN YIELD BY PROVINCE ===")

# YOUR CODE HERE: Group by province and compute mean yield
# Don't forget .compute()!
mean_yield_by_province = None  # Replace with ddf.groupby(...)[...].mean().compute()

print(mean_yield_by_province.sort_values(ascending=False))


print("\n=== TOTAL PRODUCTION BY CROP ===")

# YOUR CODE HERE: Group by crop_name and sum production_kg
total_by_crop = None  # Replace this

print(total_by_crop.sort_values(ascending=False).head(10))


# Task 3: Filter and compare
print("\n=== IMPROVED SEED ANALYSIS ===")

# Mean yield for ALL farms
overall_mean = ddf['yield_kg_per_ha'].mean().compute()

# YOUR CODE HERE: Filter to farms using improved seeds, then calculate mean yield
improved_seed_farms = None  # Filter: ddf[ddf['uses_improved_seed'] == True]
improved_mean = None  # Then calculate mean and compute

print(f"Overall mean yield: {overall_mean:,.0f} kg/ha")
print(f"Improved seed mean yield: {improved_mean:,.0f} kg/ha")
print(f"Yield advantage: {((improved_mean/overall_mean) - 1)*100:.1f}%")


# Task 4 (Optional): Performance comparison
print("\n=== PERFORMANCE COMPARISON ===")

# Time Dask
start = time.time()
result_dask = ddf.groupby('province')['yield_kg_per_ha'].mean().compute()
dask_time = time.time() - start
print(f"Dask time: {dask_time*1000:.0f} ms")

# Time Pandas (load all files)
start = time.time()
import glob
all_files = glob.glob('data/agricultural_survey/*.parquet')
df_all = pd.concat([pd.read_parquet(f) for f in all_files])
result_pandas = df_all.groupby('province')['yield_kg_per_ha'].mean()
pandas_time = time.time() - start
print(f"Pandas time: {pandas_time*1000:.0f} ms")

print(f"\nNote: For small data, Pandas may be faster due to Dask overhead.")
print(f"Dask shines when data is too large to fit in memory!")